# Bank compain data - Cleaning




The goal of this notebook is to generate cleaned data from the [dataset](https://www.kaggle.com/datasets/henriqueyamahata/bank-marketing) from Kaggle about a bank compain for fixed term deposit.

**Important**: Since the goal of the project is not to build a perfect model for predicting but more for understanding data and for insights, only important data cleaning will be performed!

## Import data

From Kaggle, the path of the dataset is directly extracted from the link of the dataset

In [13]:
import pandas as pd
import kagglehub

In [14]:
# Download latest version
path = kagglehub.dataset_download("volodymyrgavrysh/bank-marketing-campaigns-dataset")

# Import data
df = pd.read_csv(path + "/bank-additional-full.csv", sep=";")

In [15]:
# Here is an overview of the dataset
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [16]:
# The columns of the dataset
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [17]:
# The information of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

### The columns with unknown variables

From the description of the dataset in Kaggle, the 'unknown' values denote missing values.

If dropping lines with missing values would not affect the dataset, we will proceed as such!

In [18]:
print(f"The lines without any unknown values : {len(df)-len(df[df.eq('unknown').any(axis=1)])}")
print(f"The number of lines with at least one column with a missing value : {len(df[df.eq('unknown').any(axis=1)])}")

The lines without any unknown values : 30488
The number of lines with at least one column with a missing value : 10700


An imputation of missing data is enough!

### Clean the data

In [29]:
def categorize_age(df):
  """
  This function aims to encode the age into categories. This will be used later for dashboards.

  Args:
  df : pandas dataframe

  Returns:
  df : pandas dataframe
  """

  df.loc[:,'age_category'] = None

  for index, row in df.iterrows():
    if row['age'] < 20:
      df.loc[index, 'age_category'] = 'teen'

    if row['age'] >= 20 and row['age'] < 30:
      df.loc[index, 'age_category'] = 'young_adult'

    if row['age'] >= 30 and row['age'] < 40:
      df.loc[index, 'age_category'] = 'middle_aged'

    if row['age'] >= 40 and row['age'] < 50:
      df.loc[index, 'age_category'] = 'old_aged'

    if row['age'] >= 50:
      df.loc[index, 'age_category'] = 'very_old_aged'

  return df

In [30]:
# Dataset without missing datapoints
df_clean = df[~df.apply(lambda row: row.astype(str).str.lower().eq("unknown").any(), axis=1)]

# categorize age
df_clean = categorize_age(df_clean)

# Map y with 1 oer 0 instead of yes and no for the target y
df_clean['y'] = df_clean['y'].map({'yes': 1, 'no': 0})

/tmp/ipython-input-2797133527.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,'age_category'] = None
/tmp/ipython-input-2082950525.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['y'] = df_clean['y'].map({'yes': 1, 'no': 0})


In [31]:
# A quick overview
display(df_clean)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,age_category
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,very_old_aged
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,middle_aged
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,old_aged
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,very_old_aged
6,59,admin.,married,professional.course,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,very_old_aged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,1,very_old_aged
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,0,old_aged
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,0,very_old_aged
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,1,old_aged


### OneHot Encoding

In [32]:
# Apply OneHot encoding using dummies : Please remember that this is not a good method to use in case new categories or values arrive!
df_clean = pd.get_dummies(df_clean, columns=['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome'])

In [33]:
# A quick last overview
df_clean.head()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_oct,month_sep,day_of_week_fri,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_failure,poutcome_nonexistent,poutcome_success
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,False,True,False
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,False,True,False
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,False,True,False
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,False,True,False
6,59,139,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,False,True,False


### Download the new dataset

In [34]:
pd.DataFrame.to_csv(df_clean, '/content/data_cleaned.csv', index=False)